In [1]:
import torch
import time
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import scanpy as sc
import copy
from torch.utils.data import IterableDataset
from torch.utils.data import DataLoader


print("torch version:", torch.__version__)
print("cuda version:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("gpu count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("gpu name:", torch.cuda.get_device_name(0))

torch version: 2.5.1
cuda version: 12.1
cuda available: True
gpu count: 1
gpu name: NVIDIA GeForce RTX 3070


In [3]:
class PromoterOneHotEncoder:
    def __init__(self, length=400):
        self.length = length
        self.vocab = {
            "A": 0,
            "C": 1,
            "G": 2,
            "T": 3,
            "N": 4
        }
        self.num_channels = 5

    def __call__(self, seq: str):
        """
        seq: str, length <= self.length
        return: FloatTensor (length, 5)
        """
        # 初始化为 N
        one_hot = torch.zeros(self.length, self.num_channels)
        one_hot[:, 4] = 1.0  # 默认 N

        seq = seq.upper()

        for i, base in enumerate(seq[:self.length]):
            idx = self.vocab.get(base, 4)
            one_hot[i, :] = 0.0
            one_hot[i, idx] = 1.0

        return one_hot

        
class SequentialBlockDataset(IterableDataset):
    def __init__(
        self,
        promoter_file,
        scrna_file,
        cell_ids_subset=None,
        start_idx = 0
    ):
        self.promoters = pd.read_csv(promoter_file)
        self.scrna = sc.read(scrna_file, sparse=True)

        self.promoter_encoder = PromoterOneHotEncoder(length=400)
        print("Pre-encoding promoter sequences...")
        self.promoter_tensor = self._preencode_promoters()
        print("Done.")
        #self.gene_ids = gene_ids_subset
        self.use_direct_index = cell_ids_subset is None
        self.cells = cell_ids_subset or np.arange(self.scrna.n_obs)

        # 建 promoter index -> gene_id → scrna index 的映射
        self.gene2idx = {
            g: i for i, g in enumerate(self.scrna.var.gene_id)
        }
        self.promoter2expr_idx = np.empty(len(self.promoters), dtype=np.int32)
        for i, gene_id in enumerate(self.promoters["gene_id"]):
            try:
                self.promoter2expr_idx[i] = self.gene2idx[gene_id]
            except KeyError:
                raise ValueError(f"gene_id {gene_id} not found in scRNA var")
        #print(self.gene2idx)

        self.P = len(self.promoters)
        self.C = len(self.cells)
        self.total = self.P * self.C

        self.cursor = start_idx % self.total

    def _preencode_promoters(self):
        sequences = self.promoters["sequence"].values
        n = len(sequences)

        # (N, 400, 5)
        promoter_tensor = torch.zeros(
            n, 400, 5, dtype=torch.float32
        )

        for i, seq in enumerate(sequences):
            promoter_tensor[i] = self.promoter_encoder(seq)

            if i % 1000 == 0:
                print(f"  encoded {i}/{n}")

        return promoter_tensor
    
    def __len__(self):
        return self.P * self.C

    def in_getitem(self, global_idx):
        pro_i = global_idx // self.C
        cell_i = global_idx % self.C
        promoter = self.promoter_tensor[pro_i]
        #gene_id = (self.promoters["gene_id"]).iloc[pro_i]
        if self.use_direct_index:
            expr_all = self.scrna[cell_i].X
        else:
            cell_id = self.cells[cell_i]
            expr_all = self.scrna[cell_id].X

        if hasattr(expr_all, "toarray"):
            expr_all = expr_all.toarray().astype("float32").squeeze()     # (16300,)
        expr_all = torch.from_numpy(expr_all).float()
        
        target_idx = self.promoter2expr_idx[pro_i]
        y = expr_all[target_idx]
        #expr_all[target_idx] = 0.0 # mask the promoter expression
        
        return promoter, expr_all, y

    def __getitem__(self, idx):
        pro_i = idx // self.C
        cell_i = idx % self.C
        
        return self.in_getitem(pro_i,cell_i)

    def __iter__(self):
        while True:
            yield self.in_getitem(self.cursor)
            self.cursor += 1
            if self.cursor >= self.total:
                self.cursor = 0

train_dataset = SequentialBlockDataset(
    promoter_file="promoter_train.csv",
    scrna_file="integrated_data.h5ad",
)
# val_dataset = MyDataset(
#     promoter_file="promoter_val.csv",
#     scrna_file="integrated_data.h5ad",
# )

Pre-encoding promoter sequences...
  encoded 0/16308
  encoded 1000/16308
  encoded 2000/16308
  encoded 3000/16308
  encoded 4000/16308
  encoded 5000/16308
  encoded 6000/16308
  encoded 7000/16308
  encoded 8000/16308
  encoded 9000/16308
  encoded 10000/16308
  encoded 11000/16308
  encoded 12000/16308
  encoded 13000/16308
  encoded 14000/16308
  encoded 15000/16308
  encoded 16000/16308
Done.


In [5]:
# model
class SimpleGeneModel(nn.Module):
    def __init__(self, promoter_len=400, promoter_channels=5, 
                 lstm_hidden=32, expr_dim=16262):
        super().__init__()
        # promoter LSTM
        self.lstm = nn.LSTM(input_size=promoter_channels, hidden_size=lstm_hidden,
                            batch_first=True)
        # expression MLP
        self.expr_fc = nn.Linear(expr_dim, lstm_hidden)
        # output
        self.fc_out = nn.Linear(2 * lstm_hidden, 1)
        self.relu = nn.ReLU()

    def forward(self, promoter, expr):
        # promoter: (batch, 400, 5)
        # expr: (batch, 16300)
        lstm_out, _ = self.lstm(promoter)        # (batch, 400, hidden)
        lstm_out = lstm_out[:, -1, :]            # take last time step -> (batch, hidden)
        expr_out = self.relu(self.expr_fc(expr)) # (batch, hidden)
        combined = torch.cat([lstm_out, expr_out], dim=1)  # (batch, 2*hidden)
        out = self.fc_out(combined)              # (batch, 1)
        return out

def zero_promoter(promoters):
    return torch.zeros_like(promoters)
def ab_probe(model, promoters, exprs, ys, criterion):
    model.eval()

    # A: full
    out_full = model(promoters, exprs).squeeze(1)
    loss_full = criterion(out_full, ys).item()

    # B: no promoter
    promoters_zero = zero_promoter(promoters)
    out_nop = model(promoters_zero, exprs).squeeze(1)
    loss_nop = criterion(out_nop, ys).item()

    model.train()
    return loss_full, loss_nop

def evaluate(model, loader, mode, criterion, device, max_batches=100):
    """
    mode: "full" | "no_promoter" | "promoter_only"
    """
    model.eval() # transfer to validation mode, open dropout and set batch norm
    losses = []

    with torch.no_grad(): # turn off grad compute
        for i, (promoters, exprs, ys) in enumerate(loader):
            promoters = promoters.to(device)
            exprs = exprs.to(device)
            ys = ys.to(device).float()
    
            if mode == "no_promoter":
                promoters = zero_promoter(promoters)
            elif mode == "promoter_only":
                exprs = torch.zeros_like(exprs)
    
            out = model(promoters, exprs).squeeze(1)
            loss = criterion(out, ys)
            losses.append(loss.item())
    
            if i >= max_batches:
                break

    model.train()
    return sum(losses) / len(losses)

def fill_gpu_buffer(loader, device, n):
    # promoters_buf = []
    # exprs_buf = []
    # ys_buf = []

    # it = iter(dataset)
    # for _ in range(n):
    #     p, e, y = next(it)
    #     promoters_buf.append(p)
    #     exprs_buf.append(torch.from_numpy(e))
    #     ys_buf.append(y)

    # promoters_buf = torch.stack(promoters_buf).to(device)
    # exprs_buf = torch.stack(exprs_buf).to(device)
    # ys_buf = torch.tensor(ys_buf, dtype=torch.float32, device=device)
    # return promoters_buf, exprs_buf, ys_buf
    p, e, y = next(iter(loader))  # 已经是 batch 了

    return (
        p.to(device, non_blocking=True),
        e.to(device, non_blocking=True),
        y.to(device, non_blocking=True),
        #torch.tensor(y, dtype=torch.float32, device=device),
    )
def gpu_batch(promoters_buf, exprs_buf, ys_buf, batch_size):
    idx = torch.randint(
        0,
        promoters_buf.size(0),
        (batch_size,),
        device=promoters_buf.device
    )

    return (
        promoters_buf[idx],
        exprs_buf[idx],
        ys_buf[idx],
    )

In [15]:
torch.cuda.empty_cache()
GPU_BUFFER_SIZE = 4096
BATCH_SIZE = 128
steps = int(GPU_BUFFER_SIZE/BATCH_SIZE*60)
epoches = int(9e9 / GPU_BUFFER_SIZE)
print(f"epoches: {epoches}   steps: {steps}")        
torch.cuda.empty_cache()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
model = SimpleGeneModel()
model = model.to(device)
print("model on GPU ready")
train_loader = DataLoader(
    train_dataset,
    batch_size=GPU_BUFFER_SIZE,          # 尽量大
    num_workers=0,           # 视 CPU 情况
    pin_memory=True,         # GPU 必开
    drop_last=True,
)
# val_loader = DataLoader(
#     val_dataset,
#     batch_size=128,          # 尽量大
#     num_workers=0,           # 视 CPU 情况
#     pin_memory=True,         # GPU 必开
# )
print("DataLoader ready")
criterion = torch.nn.MSELoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-4,
)

probe_batch = next(iter(train_loader))
probe_promoters, probe_exprs, probe_ys = probe_batch
probe_promoters = probe_promoters.to(device)
probe_exprs = probe_exprs.to(device)
probe_ys = probe_ys.to(device).float()
probe_steps = []
probe_loss_full = []
probe_loss_nop = []
probe_gap = []
train_loss = [];validation_loss = []


model.train()
prom_buf, expr_buf, y_buf = fill_gpu_buffer(train_loader, device,GPU_BUFFER_SIZE)
for epoch in range(epoches):
    t0 = time.perf_counter()
    #prom_buf, expr_buf, y_buf = fill_gpu_buffer(train_loader, device,GPU_BUFFER_SIZE)
    next_prom_buf, next_expr_buf, next_y_buf = fill_gpu_buffer(train_loader, device,GPU_BUFFER_SIZE)
    t1 = time.perf_counter()
    print(f"buffer time: {t1-t0:.3f}s")
    print(f"load time: {epoches*(t1-t0)/3600} hours")
    #torch.cuda.synchronize()
    for step in range(steps):
        promoters, exprs, ys = gpu_batch(
            prom_buf, expr_buf, y_buf, BATCH_SIZE
        )
    
        optimizer.zero_grad()
        out = model(promoters, exprs).squeeze()
        loss = criterion(out, ys)
        loss.backward()
        optimizer.step()
    #torch.cuda.synchronize()
    t2 = time.perf_counter()
    print(f"train time: {t2-t1:.3f}s")
    prom_buf, expr_buf, y_buf = next_prom_buf, next_expr_buf, next_y_buf
    # if epoch % 2 == 0:
    #     print(loss.item())
    #     train_loss.append(loss.item())
    #     loss_A, loss_B = ab_probe(
    #         model,
    #         probe_promoters,
    #         probe_exprs,
    #         probe_ys,
    #         criterion
    #     )
    
    #     probe_steps.append(epoch/30000)
    #     probe_loss_full.append(loss_A)
    #     probe_loss_nop.append(loss_B)
    #     probe_gap.append(loss_B - loss_A)
    
    #     print(
    #         f"[A/B] step {step} | "
    #         f"full={loss_A:.4f}, noP={loss_B:.4f}, gap={loss_B-loss_A:.4f}"
    #     )

        # val_full = evaluate(
        #     model, val_loader, "full", criterion, device
        # )
        # val_no_p = evaluate(
        #     model, val_loader, "no_promoter", criterion, device
        # )
        # val_p_only = evaluate(
        #     model, val_loader, "promoter_only", criterion, device
        # )
        # validation_loss.append(val_full)
        # print(
        #     f"[VAL] step {step} | "
        #     f"full: {val_full:.4f}, "
        #     f"noP: {val_no_p:.4f}, "
        #     f"P-only: {val_p_only:.4f}"
        # )
    t3 = time.perf_counter()
    print(f"validataion time: {t3-t2:.3f}s")

plt.figure(figsize=(8, 5))
plt.plot(probe_steps, probe_loss_full, label="Full (promoter + expr)")
plt.plot(probe_steps, probe_loss_nop, label="No promoter")
plt.xlabel("Training step")
plt.ylabel("Probe loss")
plt.title("A/B probe loss during training")
plt.legend()
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(probe_steps, probe_gap, marker="o")
plt.axhline(0, linestyle="--", color="gray")
plt.xlabel("Training step")
plt.ylabel("Loss gap (no-promoter − full)")
plt.title("Promoter contribution gap")
plt.grid(True)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(probe_steps, train_loss, label="train loss")
plt.plot(probe_steps, validation_loss, label="validation loss")
plt.xlabel("Training step")
plt.ylabel("Loss")
plt.grid(True)
plt.legend()
plt.show()

epoches: 2197265   steps: 1920
Using device: cuda
model on GPU ready
DataLoader ready
buffer time: 11.081s
load time: 6763.532011697976 hours
train time: 5.994s
validataion time: 0.000s
buffer time: 11.440s
load time: 6982.659512858319 hours
train time: 6.056s
validataion time: 0.000s


KeyboardInterrupt: 